In [ ]:
import os
import fitz  # PyMuPDF
from transformers import pipeline

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Opens a PDF file and extracts all text content page by page.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"The file {pdf_path} was not found.")
    
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            page_text = page.get_text()
            if page_text:
                text += page_text + "\n"
                
    return text.strip()

def summarize_text(text: str, max_sentences: int = 10) -> str:
    """
    Uses the modern Hugging Face 'text-generation' pipeline with an instruction-tuned model
    to generate a concise summary constrained to a maximum number of sentences.
    """
    if not text:
        return "No text could be extracted from the PDF."

    print("Loading local text-generation pipeline...")
    # Use a standard open-source instruction model supported by text-generation
    summarizer = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
    
    # Construct a clear prompt enforcing the sentence limit constraint
    prompt = f"""
    Please provide a concise summary of the following text. 
    Your summary must be strictly no more than {max_sentences} sentences long.

    Text:
    {text[:4000]}  # Truncate to fit model context window if necessary
    """
    
    print("Generating summary...")
    # Format as a chat message for instruction-tuned models
    messages = [
        {"role": "system", "content": "You are a helpful assistant that summarizes documents accurately."},
        {"role": "user", "content": prompt}
    ]
    
    output = summarizer(messages, max_new_tokens=300, do_sample=False)
    
    # Extract the generated response text from the chat output structure
    summary_text = output[0]['generated_text'][-1]['content']
    
    # Post-processing: Ensure sentence count is strictly <= max_sentences
    sentences = [s.strip() for s in summary_text.split('.') if s.strip()]
    if len(sentences) > max_sentences:
        sentences = sentences[:max_sentences]
        summary_text = '. '.join(sentences) + '.'
        
    return summary_text

# --- Execution Example ---
if __name__ == "__main__":
    pdf_file_path = "/Users/pjk/IOD/CoffeeReviews.pdf"
    
    try:
        print(f"Reading text from: {pdf_file_path}")
        raw_text = extract_text_from_pdf(pdf_file_path)
        print(f"Extracted {len(raw_text)} characters of text.")
        
        # Generate summary
        summary = summarize_text(raw_text, max_sentences=10)
        
        print("\n--- PDF Summary (Max 10 Sentences) ---")
        print(summary)
        
    except Exception as e:
        print(f"An error occurred: {e}")

Reading text from: /Users/pjk/IOD/CoffeeReviews.pdf
Extracted 47850 characters of text.
Loading local text-generation pipeline...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating summary...

--- PDF Summary (Max 10 Sentences) ---
This study explores the sentiment analysis of consumer reviews from Starbucks' customers using both classical machine learning and deep learning approaches. It analyzes over 700 reviews from ConsumerAffairs, focusing on the retail coffee sector. The dataset shows a highly imbalanced distribution with predominantly negative feedback. A dual-model approach is employed, combining traditional classifiers and deep learning models like LSTM and CNN. Results indicate that SVM achieves the highest accuracy (91%) among all models, followed by Bidirectional LSTM. The study highlights the importance of balanced datasets and tailored model selection in sentiment analysis for optimizing customer experiences in the retail coffee industry.
